# cortrix-skills · LangChain demo

Wire Cortrix into a LangChain agent in one line with `as_langchain_tools(kit)`.

> **Skeleton notebook.** Cells are the canonical usage flow. A real run needs a live
> cortrix-server + an LLM API key (LangChain ReAct round-trip) and is exercised during
> integration (D3.5), not in standalone development. The unit suite mocks the SDK layer.

## Install

```bash
pip install cortrix-skills[langchain] langchain-anthropic
```

In [ ]:
from cortrix_skills import CortrixToolKit
from cortrix_skills.adapters import as_langchain_tools

kit = CortrixToolKit(
    base_url="https://cortrix.example.com",
    api_key="sk-cortrix-...",
)

tools = as_langchain_tools(kit)
print(f"{len(tools)} LangChain tools")
print([t.name for t in tools][:5], "...")

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import AgentExecutor, create_react_agent

llm = ChatAnthropic(model="claude-...")
agent = create_react_agent(llm, tools)
executor = AgentExecutor(agent=agent, tools=tools, handle_parsing_errors=True)

result = executor.invoke({"input": "find last week's notes on the P12 design"})
print(result["output"])

## Errors are machine-readable

A Cortrix error becomes a LangChain `ToolException` whose message is the four GEN-Agent
fields (`code` / `retryable` / `category` / `retry_after_ms` / `structured_data`) as JSON,
so the agent can decide whether to retry or back off on its own.